# Lesson 16 Lab — TensorRT INT4 Block Quantization: Q/DQ, Packing, and WoQ

**Puzzle:** What must be present in a graph and serialized weight buffer before TensorRT can consume INT4 weights?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

TensorRT explicit quantization represents quantization choices with Q/DQ semantics and consumes packed low-bit weights plus scales under supported block/layout constraints.

### Core mechanism

For signed INT4, two 4-bit two's-complement codes occupy one byte. Block Q/DQ applies one scale to a supported group, reconstructing floating-point values for the consuming operation or enabling a fused weight-only implementation.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "16-tensorrt-int4"
device = require_cuda()
torch.manual_seed(2026 + 16)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

A valid packer can still produce an engine-incompatible graph; a valid graph can still select a slow tactic. Semantics, serialization, build, kernel selection, and runtime are separate gates.

### What this code tests

The CUDA lab validates block Q/DQ and exact nibble round-trip while an independent package probe prevents a false TensorRT-engine claim.

**Experiment:** Perform block INT4 Q/DQ and nibble packing on CUDA, verify exact unpacking, and separately probe the TensorRT package.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import importlib.util
w=torch.randn(512,1024,device=device); q,scales,dq=symmetric_quantize(w,bits=4,group_size=64)
codes=(q.to(torch.int16)&0xF).flatten(); packed=(codes[0::2]|(codes[1::2]<<4)).to(torch.uint8)
lo=(packed.to(torch.int16)&0xF); hi=((packed.to(torch.int16)>>4)&0xF); unpack=torch.stack([lo,hi],1).flatten()
unpack=torch.where(unpack>=8,unpack-16,unpack).to(torch.int8).reshape_as(q)
result=base_result(16,"pytorch-gpu"); result.update({"shape":list(w.shape),"group_size":64,"packed_bytes":packed.numel(),
    "codes_exact_after_unpack":bool(torch.equal(q,unpack)),"qdq_error":error_metrics(w,dq),
    "tensorrt_installed":importlib.util.find_spec("tensorrt") is not None,
    "conclusion":"Block Q/DQ and nibble packing were validated; TensorRT engine execution was not inferred from the reference path."})


## 3. Inspect the evidence

Packing correctness and Q/DQ error are real; engine build and latency remain unmeasured unless TensorRT is installed and executes.

### Acceptance and rollback gate

Round-trip every packed code, verify scale axis/block size and ONNX Q/DQ placement, inspect the built engine, then benchmark the engine against the same baseline.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "codes_exact_after_unpack": true,
  "conclusion": "Block Q/DQ and nibble packing were validated; TensorRT engine execution was not inferred from the reference path.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:49+00:00",
  "group_size": 64,
  "lesson": 16,
  "packed_bytes": 262144,
  "qdq_error": {
    "cosine": 0.99425673,
    "mae": 0.09142464,
    "max_abs": 0.32329714,
    "rmse": 0.10770572
  },
  "schema_version": 1,
  "shape": [
    512,
    1024
  ],
  "tensorrt_installed": false
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Validate graph semantics, packing, scales, engine inspection, and timing as separate gates.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).